In [14]:
!nvidia-smi
!git clone https://github.com/KUO-WEL/CA_Final_Project.git
%cd CA_Final_Project

Mon Jun 15 16:40:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
%%writefile main.cu
#include <iostream>
#include <cstring>
#include <cuda_runtime.h>
#include "include/dataset.h"

#ifndef NUM_PACKETS
#define NUM_PACKETS 64
#endif

#define THREADS_PER_BLOCK 256

__global__ void cross_correlation_2d_kernel(
    const float* d_rx_all,
    const float* d_preamble,
    float* d_result_all,
    int num_windows
) {
    int window_idx = blockIdx.x * blockDim.x + threadIdx.x;
    int packet_idx = blockIdx.y;

    __shared__ float s_preamble[PREAMBLE_LEN];

    int local_tid = threadIdx.x;
    if (local_tid < PREAMBLE_LEN) {
        s_preamble[local_tid] = d_preamble[local_tid];
    }
    __syncthreads();

    if (window_idx < num_windows) {
        const float* current_rx = d_rx_all + packet_idx * RX_LEN;
        float* current_result = d_result_all + packet_idx * num_windows;

        float sum = 0.0f;
        for (int j = 0; j < PREAMBLE_LEN; ++j) {
            sum += s_preamble[j] * current_rx[window_idx + j];
        }

        current_result[window_idx] = sum;
    }
}

int main() {
    int num_windows = RX_LEN - PREAMBLE_LEN + 1;

    size_t rx_bytes = NUM_PACKETS * RX_LEN * sizeof(float);
    size_t preamble_bytes = PREAMBLE_LEN * sizeof(float);
    size_t result_bytes = NUM_PACKETS * num_windows * sizeof(float);

    float* h_rx_all = new float[NUM_PACKETS * RX_LEN];
    float* h_result_all = new float[NUM_PACKETS * num_windows];

    for (int i = 0; i < NUM_PACKETS; ++i) {
        int pattern_id = i % 8;
        memcpy(&h_rx_all[i * RX_LEN],
               &RX_SIGNALS[pattern_id * RX_LEN],
               RX_LEN * sizeof(float));
    }

    float *d_rx_all, *d_preamble, *d_result_all;
    cudaMalloc((void**)&d_rx_all, rx_bytes);
    cudaMalloc((void**)&d_preamble, preamble_bytes);
    cudaMalloc((void**)&d_result_all, result_bytes);

    cudaMemcpy(d_rx_all, h_rx_all, rx_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_preamble, PREAMBLE, preamble_bytes, cudaMemcpyHostToDevice);

    int blocks_per_packet = (num_windows + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    dim3 block(THREADS_PER_BLOCK);
    dim3 grid(blocks_per_packet, NUM_PACKETS);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    cross_correlation_2d_kernel<<<grid, block>>>(
        d_rx_all,
        d_preamble,
        d_result_all,
        num_windows
    );
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_result_all, d_result_all, result_bytes, cudaMemcpyDeviceToHost);

    int total_blocks = grid.x * grid.y;
    int total_threads = total_blocks * block.x;
    float avg_time_per_packet = ms / NUM_PACKETS;

    std::cout << "=== Part 5: GPU 2D Grid Parallelism ===" << std::endl;
    std::cout << "NUM_PACKETS: " << NUM_PACKETS << std::endl;
    std::cout << "Block size: " << block.x << " threads/block" << std::endl;
    std::cout << "Blocks per packet: " << blocks_per_packet << std::endl;
    std::cout << "Total blocks: " << total_blocks << std::endl;
    std::cout << "Total threads: " << total_threads << std::endl;
    std::cout << "Kernel execution time: " << ms << " ms" << std::endl;
    std::cout << "Average time per packet: " << avg_time_per_packet << " ms" << std::endl;
    std::cout << "Shared memory per block: " << PREAMBLE_LEN * sizeof(float) << " bytes" << std::endl;

    int error_count = 0;

    for (int p = 0; p < NUM_PACKETS; ++p) {
        float max_corr = -1e9;
        int detected_idx = -1;
        float* current_result = &h_result_all[p * num_windows];

        for (int i = 0; i < num_windows; ++i) {
            if (current_result[i] > max_corr) {
                max_corr = current_result[i];
                detected_idx = i;
            }
        }

        if (detected_idx != GROUND_TRUTH[p % 8]) {
            error_count++;
        }
    }

    if (error_count == 0) {
        std::cout << "=> GPU 2D Grid 驗證成功！"
                  << NUM_PACKETS << " 個封包全部命中！" << std::endl;
    } else {
        std::cout << "=> 驗證失敗！有 "
                  << error_count << " 個封包計算錯誤。" << std::endl;
    }

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_rx_all);
    cudaFree(d_preamble);
    cudaFree(d_result_all);

    delete[] h_rx_all;
    delete[] h_result_all;

    return 0;
}

Writing main.cu


In [ ]:
!nvcc -Xptxas -v main.cu -o main -O2 -arch=sm_75
!./main

ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z27cross_correlation_2d_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z27cross_correlation_2d_kernelPKfS0_Pfi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 67 registers, used 1 barriers, 1024 bytes smem, 380 bytes cmem[0]
ptxas info    : Compile time = 47.768 ms
=== Part 5: GPU 2D Grid Parallelism ===
同時處理封包數: 64
總啟動 Blocks 數量: 1024
Kernel 執行時間: 0.330016 ms
=> GPU 2D Grid 驗證成功！64 個封包全部命中！


In [ ]:
!ncu --set basic ./main

==PROF== Connected to process 2995 (/content/CA_Final_Project/CA_Final_Project/main)
==PROF== Profiling "cross_correlation_2d_kernel" - 0: 0%....50%....100% - 9 passes
=== Part 5: GPU 2D Grid Parallelism ===
同時處理封包數: 64
總啟動 Blocks 數量: 1024
Kernel 執行時間: 2066.13 ms
=> GPU 2D Grid 驗證成功！64 個封包全部命中！
==PROF== Disconnected from process 2995
[2995] main@127.0.0.1
  cross_correlation_2d_kernel(const float *, const float *, float *, int) (16, 64, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.96
    SM Frequency                    Mhz       584.87
    Elapsed Cycles                cycle      172,733
    Memory Throughput                 %        72.56
    DRAM Throughput                   %         2.84
    Duration                         u

In [16]:
%%bash
for p in 1 8 16 32 64
do
    echo "================ NUM_PACKETS=$p ================"
    nvcc -Xptxas -v main.cu -o main -O2 -arch=sm_75 -DNUM_PACKETS=$p
    ./main
done

================ NUM_PACKETS=1 ================
=== Part 5: GPU 2D Grid Parallelism ===
NUM_PACKETS: 1
Block size: 256 threads/block
Blocks per packet: 16
Total blocks: 16
Total threads: 4096
Kernel execution time: 0.126656 ms
Average time per packet: 0.126656 ms
Shared memory per block: 1024 bytes
=> GPU 2D Grid 驗證成功！1 個封包全部命中！
================ NUM_PACKETS=8 ================
=== Part 5: GPU 2D Grid Parallelism ===
NUM_PACKETS: 8
Block size: 256 threads/block
Blocks per packet: 16
Total blocks: 128
Total threads: 32768
Kernel execution time: 0.153216 ms
Average time per packet: 0.019152 ms
Shared memory per block: 1024 bytes
=> GPU 2D Grid 驗證成功！8 個封包全部命中！
================ NUM_PACKETS=16 ================
=== Part 5: GPU 2D Grid Parallelism ===
NUM_PACKETS: 16
Block size: 256 threads/block
Blocks per packet: 16
Total blocks: 256
Total threads: 65536
Kernel execution time: 0.192288 ms
Average time per packet: 0.012018 ms
Shared memory per block: 1024 bytes
=> GPU 2D Grid 驗證成功！16 個封包全部命中！


ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z27cross_correlation_2d_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z27cross_correlation_2d_kernelPKfS0_Pfi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 67 registers, used 1 barriers, 1024 bytes smem, 380 bytes cmem[0]
ptxas info    : Compile time = 27.825 ms
ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z27cross_correlation_2d_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z27cross_correlation_2d_kernelPKfS0_Pfi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 67 registers, used 1 barriers, 1024 bytes smem, 380 bytes cmem[0]
ptxas info    : Compile time = 27.319 ms
ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z27cross_correlation_2d_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z27cross_correlation_2d_kernelPK